In [238]:
!pip install crewai

In [239]:
from crewai import Agent, Task, Crew, Process, LLM
from pydantic import BaseModel, Field
from typing import Literal, Optional

In [240]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

llm = LLM(
    model="openai/openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.7,
)


In [241]:
class Intent(BaseModel):
    topic: Optional[str] = None
    level: str = "beginner"
    reply: str

In [242]:
coordinator = Agent(
    role="Study session coordinator",
    goal="Understand what the student wants, then get the team to deliver "
         "a clear lesson, a fair quiz and helpful feedback.",
    backstory="You are an experienced coordinator in education. You delegate "
              "work to the right teammate, check its quality and ask for fixes.",
    allow_delegation=True,
    verbose=True,
    llm=llm,
)

In [243]:
explainer = Agent(
    role = "Study session explainer",
    goal = "Teach the {topic} and according to his {level}"
    "Use some good and simple example based on his {level}",
    backstory="You are a empathetic teacher who knows students level, teach them according to it with simple examples",
    allow_delegation=False,
    verbose=True,
    llm=llm,
)

In [244]:
evaluation_agent = Agent(
    role="Evaluator",
    goal="Grade the student's quiz answers fairly and explain every mistake simply.",
    backstory="You are a kind but honest examiner. You give partial credit when "
              "an answer is partly right, and you always explain why an answer "
              "is wrong in one or two simple sentences.",
    allow_delegation=False,
    verbose=True,
    llm=llm,
)

In [245]:
class Question(BaseModel):
    id: int
    type: Literal["mcq", "short"]
    question: str
    options: Optional[list[str]] = None
    correct_answer: str
    concept: str

class Quiz(BaseModel):
    topic: str
    level: str
    questions: list[Question]

In [246]:
class QuestionResult(BaseModel):
    id: int
    correct: bool
    score: float
    feedback: str

class Feedback(BaseModel):
    total_score: float
    max_score: int
    results: list[QuestionResult]
    weak_areas: list[str]
    encouragement: str

In [247]:
understand_task = Task(
    description=(
        "You are chatting with a student."
        "Student says: \"{message}\""
        "Decide the action:"
        "- learn: they named a clear, teachable topic. Fill topic and level."
        "- clarify: topic too broad or unclear. Reply with 3 small topic options."
        "- smalltalk: greeting or chit-chat. Reply friendly and ask what to learn."
        "- quit: they want to stop."
        "Write a short, friendly 'reply' in chat style."
    ),
    expected_output="The action, topic, level and a short reply.",
    llm=llm,
    agent=coordinator,
    output_pydantic=Intent,
)

In [248]:
explain_task = Task(
    description="Teach {topic} to a {level} student in under 250 words.",
    expected_output="A great, clear lesson with one example.",
  )

quiz_task = Task(
    description=(
        "if {num_questions} is not mentioned choose any of them from 3 to 5"
        "Using the lesson, write {num_questions} practice questions on {topic} "
        "for a {level} student."
        "- Mix types: mostly 'mcq' with 4 options, plus 1 'short' answer"
        "- For mcq, correct_answer must be exactly one of the options."
        "- Only ask about what the lesson taught."
        "- Tag each question with the concept it tests."
        "- Previous weak areas for this student: {weak_areas}. "
        "Include at least one question on them if relevant."
    ),
    expected_output="A quiz with questions, options, correct answers and concepts.",
    context=[explain_task],
    output_pydantic=Quiz,
    llm=llm
)

In [249]:
evaluation_task = Task(
    description=(
        "Grade this quiz on {question_answer} and student answer {student_answer}."
        "Rules:\n"
        "- Give 1 for correct, 0.5 for partly correct short answers, 0 for wrong."
        "- For mcq, the answer is written or option text."
        "- Explain each mistake in 1-2 simple sentences."
        "- List the concepts the student got wrong as weak_areas."
        "- End with one short line of encouragement."
        "Return only valid JSON."
    ),
    expected_output="Per-question results, total score, weak areas and encouragement.",
    agent=evaluation_agent,
    output_pydantic=Feedback,
)

In [250]:
crew = Crew(agents=[coordinator], tasks=[understand_task],
                process=Process.sequential)

In [251]:
result = await crew.kickoff_async(inputs={"message": "I want to learn basic php"})

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Study session coordinator                                                                               │
│                                                                                                                 │
│  Task: You are chatting with a student.Student says: "I want to learn basic php"Decide the action:- learn:      │
│  they named a clear, teachable topic. Fill topic and level.- clarify: topic too broad or unclear. Reply with 3  │
│  small topic options.- smalltalk: greeting or chit-chat. Reply friendly and ask what to learn.- quit: they      │
│  want to stop.Write a short, friendly 'reply' in chat style.                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Study session coordinator                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  topic='basic php' level='beginner' reply="Great! Let's get started with basic PHP. Which part would you like   │
│  to focus on—variables, loops, or functions?"                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [252]:
crew2 = Crew(agents=[explainer, quiz_master], tasks=[explain_task, quiz_task],
                process=Process.hierarchical,   manager_agent=coordinator,
    memory=True,
    verbose=True)

In [253]:
result2 = await crew2.kickoff_async(
    inputs={
       "topic" : intent.topic,
       "level" : intent.level,
       "weak_areas" : "Nothing",
       "num_questions" : 5,
    }
)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 8118f26c-38ee-4b18-9a0a-ad8ac4447321                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Teach basic PHP to a beginner student in under 250 words.                                                │
│  ID: 5e748840-28b6-4168-9ae5-8bf7d1f30691                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Study session coordinator                                                                               │
│                                                                                                                 │
│  Task: Teach basic PHP to a beginner student in under 250 words.                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'context': 'Teach basic PHP to a beginner student in under 250 words. Provide a great, clear lesson     │
│  with one example. Must return the actual complete content as final answer, not a summary.', 'cowork...         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Study session explainer                                                                                 │
│                                                                                                                 │
│  Task: Create a concise PHP lesson (<=250 words) with a single example, suitable for a beginner.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


... (20 outputs left out)

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Study session coordinator                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "topic": "PHP Basics for Beginners",                                                                         │
│    "level": "Beginner",                                                                                         │
│    "questions": [                                                                                               │
│      {                                                                                                          │
│        "id": 1,                                                                                                 │
│        "type": "mcq",                                                                                           │
│        "question": "Which of the following tags encloses PHP code?",                                            │
│        "options": [                                                                                             │
│          "<?php ... ?>",                                                                                        │
│          "<php ... >",                                                                                          │
│          "<?php ... >",                                                                                         │
│          "<php ... ?>"                                                                                          │
│        ],                                                                                                       │
│        "correct_answer": "<?php ... ?>",                                                                        │
│        "concept": "PHP Tags"                                                                                    │
│      },                                                                                                         │
│      {                                                                                                          │
│        "id": 2,                                                                                                 │
│        "type": "mcq",                                                                                           │
│        "question": "What is the most common way to output text to the browser in PHP?",                         │
│        "options": [                                                                                             │
│          "print",                                                                                               │
│          "echo",                                                                                                │
│          "output",                                                                                              │
│          "display"                                                                                              │
│        ],                                                                                                       │
│        "correct_answer": "echo",                                                                                │
│        "concept": "Echo"                                                                                        │
│      },                                                

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: if 5 is not mentioned choose any of them from 3 to 5Using the lesson, write 5 practice questions on      │
│  basic PHP for a beginner student.- Mix types: mostly 'mcq' with 4 options, plus 1 'short' answer- For mcq,     │
│  correct_answer must be exactly one of the options.- Only ask about what the lesson taught.- Tag each question  │
│  with the concept it tests.- Previous weak areas for this student: Nothing. Include at least one question on    │
│  them if relevant.                                                                                              │
│  Agent: Study session coordinator                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 8118f26c-38ee-4b18-9a0a-ad8ac4447321                                                                       │
│  Final Output: {                                                                                                │
│    "topic": "PHP Basics for Beginners",                                                                         │
│    "level": "Beginner",                                                                                         │
│    "questions": [                                                                                               │
│      {                                                                                                          │
│        "id": 1,                                                                                                 │
│        "type": "mcq",                                                                                           │
│        "question": "Which of the following tags encloses PHP code?",                                            │
│        "options": [                                                                                             │
│          "<?php ... ?>",                                                                                        │
│          "<php ... >",                                                                                          │
│          "<?php ... >",                                                                                         │
│          "<php ... ?>"                                                                                          │
│        ],                                                                                                       │
│        "correct_answer": "<?php ... ?>",                                                                        │
│        "concept": "PHP Tags"                                                                                    │
│      },                                                                                                         │
│      {                                                                                                          │
│        "id": 2,                                                                                                 │
│        "type": "mcq",                                                                                           │
│        "question": "What is the most common way to output text to the browser in PHP?",                         │
│        "options": [                                                                                             │
│          "print",                                                                                               │
│          "echo",                                                                                                │
│          "output",                                                                                              │
│          "display"                                                                                              │
│        ],                                                                                                       │
│        "correct_answer": "echo",                                                                                │
│        "concept": "Echo"                                                                                        │
│      },                                               

In [254]:
result2.tasks_output[0].raw

'**PHP Basics for Beginners** (under 250 words)\n\nPHP is a server‑side language that runs inside a web server. Put your code inside `<?php … ?>` tags; anything outside is sent to the browser as plain HTML. The most common way to show text is with `echo`. Variables start with `$` and can hold strings, numbers, etc.\n\nExample – a simple “Hello, World” that greets a user by name:\n\n```php\n<?php\n// Get name from query string, default to \'Guest\'\n$name = isset($_GET[\'name\']) ? htmlspecialchars($_GET[\'name\']) : \'Guest\';\n\n// Build greeting\n$greeting = "Hello, " . $name . "! Welcome to PHP.";\n\n// Output\necho $greeting;\n?>\n```\n\nSave as `greet.php` and visit `http://localhost/greet.php?name=Alice`. You’ll see “Hello, Alice! Welcome to PHP.” If you omit `?name=`, the page shows “Hello, Guest! Welcome to PHP.”\n\n**Key points**  \n1. `<?php … ?>` encloses PHP code.  \n2. `echo` prints to the browser.  \n3. Variables start with `$`.  \n4. Use `.` to concatenate strings.  \n5.

In [255]:
from IPython.display import display, Markdown

display(Markdown(result2.tasks_output[0].raw))

**PHP Basics for Beginners** (under 250 words)

PHP is a server‑side language that runs inside a web server. Put your code inside `<?php … ?>` tags; anything outside is sent to the browser as plain HTML. The most common way to show text is with `echo`. Variables start with `$` and can hold strings, numbers, etc.

Example – a simple “Hello, World” that greets a user by name:

```php
<?php
// Get name from query string, default to 'Guest'
$name = isset($_GET['name']) ? htmlspecialchars($_GET['name']) : 'Guest';

// Build greeting
$greeting = "Hello, " . $name . "! Welcome to PHP.";

// Output
echo $greeting;
?>
```

Save as `greet.php` and visit `http://localhost/greet.php?name=Alice`. You’ll see “Hello, Alice! Welcome to PHP.” If you omit `?name=`, the page shows “Hello, Guest! Welcome to PHP.”

**Key points**  
1. `<?php … ?>` encloses PHP code.  
2. `echo` prints to the browser.  
3. Variables start with `$`.  
4. Use `.` to concatenate strings.  
5. Sanitize user input (`htmlspecialchars`) to avoid XSS.

Try changing the name or adding more variables to practice.

In [256]:
quiz   = result2.tasks_output[1].pydantic

In [257]:
answers_by_student=[];

for question in quiz.questions:
  print(question.question)
  if question.options:
    for ind, option in enumerate(question.options):
      print(f"({ind}) {option}")
    print("Write option number (e.g. 0): ")
    answer = input().strip()
    selected_index = int(answer)
    while selected_index < 0 or selected_index >= len(question.options):
      print("Invalid option number. Please try again.")
      answer = input().strip()
      selected_index = int(answer)
    answers_by_student.append(question.options[selected_index])
  else:
    print("Write answer: ")
    answer = input().strip()
    answers_by_student.append(answer)
  print()

Which of the following tags encloses PHP code?
(0) <?php ... ?>
(1) <php ... >
(2) <?php ... >
(3) <php ... ?>
Write option number (e.g. 0): 
0

What is the most common way to output text to the browser in PHP?
(0) print
(1) echo
(2) output
(3) display
Write option number (e.g. 0): 
1

How do you start a variable name in PHP?
(0) $
(1) #
(2) @
(3) %
Write option number (e.g. 0): 
2

Which operator is used to concatenate strings in PHP?
(0) +
(1) .
(2) *
(3) &
Write option number (e.g. 0): 
0

What function is used in the example to sanitize user input and avoid XSS?
Write answer: 
i don't know



In [258]:
eval_crew = Crew(
    agents=[evaluation_agent],
    tasks=[evaluation_task],
    process=Process.hierarchical,
    manager_agent=coordinator,
    memory=False,
    verbose=True,
)

serialized_questions = [q.model_dump() for q in quiz.questions]

result3 = await eval_crew.kickoff_async(
    inputs={
        "question_answer": serialized_questions,
        "student_answer": answers_by_student,
    }
)
feedback = result3.pydantic

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 66b60595-e51d-493d-8db3-a44e704487dc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Grade this quiz on [{'id': 1, 'type': 'mcq', 'question': 'Which of the following tags encloses PHP       │
│  code?', 'options': ['<?php ... ?>', '<php ... >', '<?php ... >', '<php ... ?>'], 'correct_answer': '<?php ...  │
│  ?>', 'concept': 'PHP Tags'}, {'id': 2, 'type': 'mcq', 'question': 'What is the most common way to output text  │
│  to the browser in PHP?', 'options': ['print', 'echo', 'output', 'display'], 'correct_answer': 'echo',          │
│  'concept': 'Echo'}, {'id': 3, 'type': 'mcq', 'question': 'How do you start a variable name in PHP?',           │
│  'options': ['$', '#', '@', '%'], 'correct_answer': '$', 'concept': 'Variables'}, {'id': 4, 'type': 'mcq',      │
│  'question': 'Which operator is used to concatenate strings in PHP?', 'options': ['+', '.', '*', '&'],          │
│  'correct_answer': '.', 'concept': 'String Concatenation'}, {'id': 5, 'type': 'short', 'question': 'What        │
│  function is used in the example to sanitize user input and avoid XSS?', 'options': None, 'correct_answer':     │
│  'htmlspecialchars', 'concept': 'User Input Sanitization'}] and student answer ['<?php ... ?>', 'echo', '@',    │
│  '+', "i don't know"].Rules:                                                                                    │
│  - Give 1 for correct, 0.5 for partly correct short answers, 0 for wrong.- For mcq, the answer is written or    │
│  option text.- Explain each mistake in 1-2 simple sentences.- List the concepts the student got wrong as        │
│  weak_areas.- End with one short line of encouragement.Return only valid JSON.                                  │
│  ID: 2d0ca957-3a98-404e-9d61-48d7befc96ba                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Study session coordinator                                                                               │
│                                                                                                                 │
│  Task: Grade this quiz on [{'id': 1, 'type': 'mcq', 'question': 'Which of the following tags encloses PHP       │
│  code?', 'options': ['<?php ... ?>', '<php ... >', '<?php ... >', '<php ... ?>'], 'correct_answer': '<?php ...  │
│  ?>', 'concept': 'PHP Tags'}, {'id': 2, 'type': 'mcq', 'question': 'What is the most common way to output text  │
│  to the browser in PHP?', 'options': ['print', 'echo', 'output', 'display'], 'correct_answer': 'echo',          │
│  'concept': 'Echo'}, {'id': 3, 'type': 'mcq', 'question': 'How do you start a variable name in PHP?',           │
│  'options': ['$', '#', '@', '%'], 'correct_answer': '$', 'concept': 'Variables'}, {'id': 4, 'type': 'mcq',      │
│  'question': 'Which operator is used to concatenate strings in PHP?', 'options': ['+', '.', '*', '&'],          │
│  'correct_answer': '.', 'concept': 'String Concatenation'}, {'id': 5, 'type': 'short', 'question': 'What        │
│  function is used in the example to sanitize user input and avoid XSS?', 'options': None, 'correct_answer':     │
│  'htmlspecialchars', 'concept': 'User Input Sanitization'}] and student answer ['<?php ... ?>', 'echo', '@',    │
│  '+', "i don't know"].Rules:                                                                                    │
│  - Give 1 for correct, 0.5 for partly correct short answers, 0 for wrong.- For mcq, the answer is written or    │
│  option text.- Explain each mistake in 1-2 simple sentences.- List the concepts the student got wrong as        │
│  weak_areas.- End with one short line of encouragement.Return only valid JSON.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Study session coordinator                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "total_score": 2,                                                                                            │
│    "max_score": 5,                                                                                              │
│    "results": [                                                                                                 │
│      {                                                                                                          │
│        "id": 1,                                                                                                 │
│        "correct": true,                                                                                         │
│        "score": 1,                                                                                              │
│        "feedback": "Correct."                                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "id": 2,                                                                                                 │
│        "correct": true,                                                                                         │
│        "score": 1,                                                                                              │
│        "feedback": "Correct."                                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "id": 3,                                                                                                 │
│        "correct": false,                                                                                        │
│        "score": 0,                                                                                              │
│        "feedback": "You answered '@', but the correct symbol is '$'."                                           │
│      },                                                                                                         │
│      {                                                                                                          │
│        "id": 4,                                                                                                 │
│        "correct": false,                                                                                        │
│        "score": 0,                                                                                              │
│        "feedback": "You answered '+', but the correct operator for concatenation is '.'."                       │
│      },                                                                                                         │
│      {                                                                                                          │
│        "id": 5,                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Grade this quiz on [{'id': 1, 'type': 'mcq', 'question': 'Which of the following tags encloses PHP       │
│  code?', 'options': ['<?php ... ?>', '<php ... >', '<?php ... >', '<php ... ?>'], 'correct_answer': '<?php ...  │
│  ?>', 'concept': 'PHP Tags'}, {'id': 2, 'type': 'mcq', 'question': 'What is the most common way to output text  │
│  to the browser in PHP?', 'options': ['print', 'echo', 'output', 'display'], 'correct_answer': 'echo',          │
│  'concept': 'Echo'}, {'id': 3, 'type': 'mcq', 'question': 'How do you start a variable name in PHP?',           │
│  'options': ['$', '#', '@', '%'], 'correct_answer': '$', 'concept': 'Variables'}, {'id': 4, 'type': 'mcq',      │
│  'question': 'Which operator is used to concatenate strings in PHP?', 'options': ['+', '.', '*', '&'],          │
│  'correct_answer': '.', 'concept': 'String Concatenation'}, {'id': 5, 'type': 'short', 'question': 'What        │
│  function is used in the example to sanitize user input and avoid XSS?', 'options': None, 'correct_answer':     │
│  'htmlspecialchars', 'concept': 'User Input Sanitization'}] and student answer ['<?php ... ?>', 'echo', '@',    │
│  '+', "i don't know"].Rules:                                                                                    │
│  - Give 1 for correct, 0.5 for partly correct short answers, 0 for wrong.- For mcq, the answer is written or    │
│  option text.- Explain each mistake in 1-2 simple sentences.- List the concepts the student got wrong as        │
│  weak_areas.- End with one short line of encouragement.Return only valid JSON.                                  │
│  Agent: Study session coordinator                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [259]:
print(f"Score: {feedback.total_score}/{feedback.max_score}\n")
for r in feedback.results:
    print(("✔" if r.correct else "✘"), f"Q{r.id}:", r.feedback)
print("\n\n Work on:", ", ".join(feedback.weak_areas) or "nothing, great job!")
print("", feedback.encouragement)

Score: 2.0/5

✔ Q1: Correct.
✔ Q2: Correct.
✘ Q3: You answered '@', but the correct symbol is '$'.
✘ Q4: You answered '+', but the correct operator for concatenation is '.'.
✘ Q5: You answered 'i don't know', but the correct function is 'htmlspecialchars'.


 Work on: Variables, String Concatenation, User Input Sanitization
 Keep up the good effort!
